In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib import animation

from IPython.display import HTML

import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

import os
import cmocean

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import contextily as ctx

import glob
from PIL import Image
import imageio.v3 as iio
import scienceplots

In [ ]:
# Configuration (do not containerize this cell)
import minio

param_minio_endpoint = "scruffy.lab.uvalight.net:9000"
param_minio_user_prefix = "zhanqing2016@gmail.com"  # Your personal folder in the naa-vre-user-data bucket in MinIO
secret_minio_access_key = "sFmE1jsm5hjJBBGh5RBL"
secret_minio_secret_key = "pczCG6FRpXQEtad7lAvXv00iCYFd5Dpa1g8GOWzR"


In [ ]:
# Cell-1-parameter
# Access MinIO files
from minio import Minio
mc = Minio(endpoint=param_minio_endpoint,
           access_key=secret_minio_access_key,
           secret_key=secret_minio_secret_key)

# List existing buckets: get a list of all available buckets
mc.list_buckets()

In [ ]:
from pathlib import Path
import os
import glob
import xarray as xr

In [ ]:
param_variable = "Y2c"

# Time period to extract (YYYY-MM, inclusive)
param_START = "2015-01"
param_END   = "2015-12"


In [ ]:
# cell-par-DT
# Choose variable and time period → export as NetCDF in original model format


variable       = param_variable


# 2D reference variables always included alongside the selected variable
ALWAYS_KEEP = ["bathymetry", "latc", "lonc","elev"]
base_dir = "/home/jovyan/Cloud Storage/naa-vre-waddenzee-shared/dws/model_output/archived_runs"
# Output directory for exported NetCDF files
OUTPUT_NC_DIR = Path(base_dir).parent.parent / "results" / "DT_import"
OUTPUT_NC_DIR.mkdir(parents=True, exist_ok=True)

# Only spinup_10
spinup_dir = os.path.join(base_dir, "spinup_10")

def _yyyymm_from_path(path: str) -> str:
    """YYYY-MM from filename pattern dws_500m.3d.YYYYMM.nc"""
    yyyymm = os.path.basename(path).split(".")[2]
    return f"{yyyymm[:4]}-{yyyymm[4:]}"

all_files = sorted(glob.glob(os.path.join(spinup_dir, "dws_500m.3d.*.nc")))
selected  = [f for f in all_files if param_START <= _yyyymm_from_path(f) <= param_END]

if not selected:
    raise FileNotFoundError(
        f"No files found in {spinup_dir} for {param_START}–{param_END}"
    )
print(f"Selected {len(selected)} file(s): {[os.path.basename(f) for f in selected]}")

# Concatenate files; preserve original structure (coords, attrs, encoding)
ds_multi = xr.open_mfdataset(
    selected,
    combine="nested",
    concat_dim="time",
    data_vars="minimal",
    coords="minimal",
    compat="override",
    join="override",
)

# Keep selected variable + mandatory 2D reference variables
export_vars = [variable] + [v for v in ALWAYS_KEEP if v in ds_multi.variables]
missing = [v for v in ALWAYS_KEEP if v not in ds_multi.variables]
if missing:
    print(f"Warning: these ALWAYS_KEEP variables were not found and will be skipped: {missing}")

ds_export = ds_multi[export_vars]

output_path = OUTPUT_NC_DIR / f"spinup_10_{variable}_{param_START}_{param_END}.nc"

# Preserve float32 precision and apply light compression for all exported variables
encoding = {v: {"dtype": "float32", "zlib": True, "complevel": 4} for v in export_vars}
ds_export.to_netcdf(output_path, encoding=encoding)

print(f"\nSaved : {output_path}")
print(f"Vars  : {export_vars}")
print(f"Time  : {str(ds_export.time.values[0])[:10]} → {str(ds_export.time.values[-1])[:10]}")
print(f"Shape : {dict(ds_export.dims)}")
